<a href="https://colab.research.google.com/github/human720/Booking-uas-/blob/main/Booking_(uas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# PROSES BOOKING
# =========================
def buat_booking(user):
    lihat_fasilitas()
    try:
        id_fasilitas = int(input("\nPilih ID fasilitas : "))
    except ValueError:
        print("ID fasilitas harus berupa angka.")
        return

    tanggal = input("Tanggal (DD-MM-YYYY) : ").strip()
    mulai = input("Jam mulai (HH:MM)     : ").strip()
    selesai = input("Jam selesai (HH:MM)   : ").strip()

    hasil, pesan = validasi_input_jadwal(tanggal, mulai, selesai)
    if pesan:
        print(pesan)
        return

    with koneksi() as conn:
        cur = conn.cursor()
        cur.execute("""
            SELECT nama_fasilitas, harga_per_jam
            FROM fasilitas WHERE id = ?
        """, (id_fasilitas,))
        fasilitas = cur.fetchone()

    if not fasilitas:
        print("Fasilitas tidak ditemukan.")
        return

    if cek_bentrok(id_fasilitas, tanggal, mulai, selesai):
        print("Jadwal sudah digunakan. Booking ditolak.")
        rekomendasi_slot(id_fasilitas, tanggal)
        return

    nama_penyewa = input("Nama penyewa       : ").strip()
    no_hp = input("Nomor HP            : ").strip()
    catatan = input("Catatan tambahan    : ").strip()

    if not nama_penyewa or not no_hp:
        print("Nama penyewa dan nomor HP wajib diisi.")
        return

    durasi = int(mulai[:2])
    durasi_selesai = int(selesai[:2])
    jumlah_jam = durasi_selesai - durasi
    total = jumlah_jam * fasilitas[1]

    with koneksi() as conn:
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO booking
            (kode_booking, id_fasilitas, id_user, tanggal, jam_mulai,
             jam_selesai, nama_penyewa, no_hp, catatan, total_harga,
             status_booking, status_bayar)
            VALUES ('PENDING', ?, ?, ?, ?, ?, ?, ?, ?, ?,
                    'MENUNGGU PEMBAYARAN', 'BELUM LUNAS')
        """, (id_fasilitas, user["id"], tanggal, mulai, selesai,
              nama_penyewa, no_hp, catatan, total))
        booking_id = cur.lastrowid
        kode = f"BK{booking_id:04d}"
        cur.execute("UPDATE booking SET kode_booking = ? WHERE id = ?",
                    (kode, booking_id))
        conn.commit()

    print("\nBooking berhasil dibuat!")
    print(f"Kode booking : {kode}")
    print(f"Fasilitas    : {fasilitas[0]}")
    print(f"Total harga  : Rp{total:,}")
    print("Status       : MENUNGGU PEMBAYARAN")


def riwayat_booking(user):
    print("\n=== RIWAYAT BOOKING ===")
    with koneksi() as conn:
        cur = conn.cursor()
        cur.execute("""
            SELECT b.kode_booking, f.nama_fasilitas, b.tanggal,
                   b.jam_mulai, b.jam_selesai, b.status_booking,
                   b.status_bayar, b.total_harga
            FROM booking b
            JOIN fasilitas f ON b.id_fasilitas = f.id
            WHERE b.id_user = ?
            ORDER BY b.id DESC
        """, (user["id"],))
        data = cur.fetchall()

    if not data:
        print("Belum ada riwayat booking.")
        return

    for row in data:
        print("-" * 55)
        print(f"Kode       : {row[0]}")
        print(f"Fasilitas  : {row[1]}")
        print(f"Tanggal    : {row[2]} ({row[3]}-{row[4]})")
        print(f"Status     : {row[5]}")
        print(f"Pembayaran : {row[6]}")
        print(f"Total      : Rp{row[7]:,}")